In [1]:
from copy import deepcopy
import datetime
import glob
import os
import pickle
import sys
import numpy as np
import pandas as pd
import h5py
import scipy.io as sio

In [2]:
def get_grass_start_end_time(starttime_raw, endtime_raw):
    
    time_str_elements = starttime_raw.flatten()
    start_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))
    time_str_elements = endtime_raw.flatten()
    end_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))

    start_time = start_time.split(':')
    second_elements = start_time[-1].split('.')
    start_time = datetime.datetime(1990,1,1,hour=int(float(start_time[0])), minute=int(float(start_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))
    end_time = end_time.split(':')
    second_elements = end_time[-1].split('.')
    end_time = datetime.datetime(1990,1,1,hour=int(float(end_time[0])), minute=int(float(end_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))

    return start_time, end_time

In [3]:
def check_load_mgh_dataset(signal_path, label_path, channels=None, report_and_actual_time_tol=300, reverse_sign=False):

    gender = None
    handedness = None
    
    try: # usually Grass, saved as pre 7.3 format:
        ff = sio.loadmat(signal_path)
        data_path = os.path.basename(signal_path)
        if 's' not in ff:
            raise Exception('No signal found in %s.'%data_path)
        signal = ff['s']
        if reverse_sign:
            signal = -signal
        channel_names = [ff['hdr'][0,i]['signal_labels'][0].upper().replace('EKG','ECG') for i in range(ff['hdr'].shape[1])]
        if 'grass' in signal_path:
            Fs = 200.
        else:
            raise Exception('Safety check to make sure this is Grass data with Fs=200')
        # Label part for grass:
        if 'grass' in signal_path:
            with h5py.File(label_path) as ffl:
                sleep_stage = ffl['stage'][()].flatten()
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())


    except: # saved as .mat 7.3. new grass or natus files:
        with h5py.File(signal_path,'r') as ff:

            hdr = ff['hdr']
            signal_labels = hdr['signal_labels'][:]
            channel_names = [ ''.join(list(map(chr, ff[signal_labels[i,0]][:]))) for i in range(signal_labels.shape[0]) ]
            channel_names = [channel.upper() for channel in channel_names]
            signal = ff['s'][()]
            signal = np.transpose(signal);

            if 'recording' in ff.keys(): # only available for natus:
                year = int(np.squeeze(ff['recording']['year']))
                month = int(np.squeeze(ff['recording']['month']))
                day = int(np.squeeze(ff['recording']['day']))
                hour = int(np.squeeze(ff['recording']['hour']))
                if (hour >= 7) and (hour <=10):         # 'typo' by sleep techs
                    hour = hour + 12
                minute = int(np.squeeze(ff['recording']['minute']))
                second = int(np.squeeze(ff['recording']['second']))
                millisecond = int(np.squeeze(ff['recording']['millisecond']))
                Fs = int(np.squeeze(ff['recording']['samplingrate']))

                start_time = datetime.datetime(1990,1,1,hour=hour, minute=minute,
                        second=second, microsecond=int(millisecond*1000))
                end_time = start_time+datetime.timedelta(seconds=max(signal.shape)/Fs)

            else: # grass:
                if 'grass' in signal_path:
                    Fs = 200.
                else:
                    raise Exception('Safety check to make sure this is Grass data with Fs=200')

            # Label part for grass:
            ff.close()

        # load labels
        with h5py.File(label_path) as ffl:
            sleep_stage = ffl['stage'][()].flatten()
            if 'grass' in signal_path:
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())
            ffl.close()

    # end of loading part
    ##################################
    
    # check signal length = sleep stage length
    if sleep_stage.shape[0]!=signal.shape[1]:
        raise Exception('Inconsistent sleep stage length (%d) and signal length (%d) in %s'%(sleep_stage.shape[0],signal.shape[1],data_path))

    # only take signal channels to study
    if channels is None:
        signal_channel_ids = list(range(len(channel_names)))
        signal_channel_names = channel_names
        
    elif 'SumEffort' in channels:
        signal_channel_ids = []
        signal_channel_names = []
        for ichannel in ['ABD', 'CHEST']:
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==ichannel.upper():
                    signal_channel_ids.append(j)
                    signal_channel_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%ichannel)
        signal = signal[signal_channel_ids,:]#.T
        # do effort belt average here:
        signal = np.sum(signal,0,keepdims=1)/2
        
    else:
        signal_channel_ids = []
        signal_channel_names = []
        for i in range(len(channels)):
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==channels[i].upper():
                    signal_channel_ids.append(j)
                    signal_channel_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%channels[i])
        signal = signal[signal_channel_ids,:]#.T

    # check whether the signal contains NaN
    if np.any(np.isnan(signal)):
        raise Exception('Found Nan in signal in %s'%data_path)

    # check whether sleep_stage contains all 5 stages
    stages = np.unique(sleep_stage[np.logical_not(np.isnan(sleep_stage))]).astype(int).tolist()
    if len(stages)<=1:
        raise Exception('#sleep stage <= 1: %s in %s'%(stages,data_path))

    params = {'Fs':Fs*1.0, 'channel_ids': signal_channel_ids, 'channel_names': signal_channel_names}
    if gender is not None:
        params['gender'] = gender
    if handedness is not None:
        params['handedness'] = handedness
    return signal, sleep_stage, params


In [4]:
signal_path = '/media/mad3/Datasets_ConvertedData/sleeplab/grass_data/Hastings_Bertie_032410_2004.000/Signal_TwinData3_818.mat'
label_path = '/media/mad3/Datasets_ConvertedData/sleeplab/grass_data/Hastings_Bertie_032410_2004.000/Labels_TwinData3_818.mat'
signal, sleep_stage, params = check_load_mgh_dataset(signal_path, label_path)

In [5]:
print(signal.shape)
print(sleep_stage.shape)
print(params)

(23, 7205000)
(7205000,)
{'Fs': 200.0, 'channel_ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22], 'channel_names': ['F3-M2', 'F4-M1', 'C3-M2', 'C4-M1', 'O1-M2', 'O2-M1', 'E1-M2', 'E2-M1', 'CHIN1-CHIN2', 'LAT', 'RAT', 'SNORE', 'CPRES', 'CFLOW', 'AIRFLOW', 'PTAF', 'LEAK', 'CHEST', 'ABD', 'SAO2', 'IC', 'HR', 'ECG']}


In [ ]:
### ignore cells below

In [11]:
sleeplab_table = pd.read_csv('sleeplab_table.csv')

In [12]:
signalfilepath = os.path.join(sleeplab_table.iloc[0].Path, sleeplab_table.iloc[0].Signalfile)
labelfilepath = os.path.join(sleeplab_table.iloc[0].Path, sleeplab_table.iloc[0].Labelfile)

In [13]:
signalfilepath = '/media/mad3/Datasets_ConvertedData/sleeplab/grass_data/Anatasia_Lori_091611_2209.000/Signal_TwinData5_Exported_34.mat'

In [14]:
i=22000
signalfilepath = os.path.join(sleeplab_table.iloc[i].Path, sleeplab_table.iloc[i].Signalfile)
labelfilepath = os.path.join(sleeplab_table.iloc[i].Path, sleeplab_table.iloc[i].Labelfile)

In [15]:
signalfilepath

'/media/mad3/Datasets_ConvertedData/sleeplab/natus_data/Jastrem~ Elain_b81f900f-a42b-423e-a9ca-9d30489ee58b/Signal_Jastrem~ Elain_b81f900f-a42b-423e-a9ca-9d30489ee58b.mat'